In [ ]:
import deeptrack as dt
import numpy as np
import matplotlib.pyplot as plt


particle = dt.Sphere(
    position=np.array([0.5, 0.5]) * 64, position_unit="pixel",
    radius=1 , refractive_index=1.45, z = 0,
)

# particle = dt.PointParticle(
#     position=np.array([0.5, 0.5]) * 64, position_unit="pixel",
#     radius=500 * dt.units.nm, refractive_index=1.45 + 0.02j,
# )


brightfield_microscope = dt.Brightfield(
    wavelength=500 * 1E-9, NA=1.0, resolution=2E-7,
    magnification=2, refractive_index_medium=1.33,
    output_region=(0, 0, 64, 64),
)

imaged_scatterer = brightfield_microscope(particle)
image = imaged_scatterer()

fig =  plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray")
plt.show()


imaged_scatterer.print_dependencies_tree()

ImportError: cannot import name 'TORCH_AVAILABLE' from partially initialized module 'deeptrack' (most likely due to a circular import) (/Users/841602/Documents/GitHub/DeepTrack2/deeptrack/__init__.py)

In [2]:
particle

Sphere(len=1, action=<lambda>)

In [3]:
scatterer.properties()

NameError: name 'scatterer' is not defined

In [ ]:
from typing import Any, TYPE_CHECKING
from deeptrack.backend.units import (
    ConversionTable,
    create_context,
    get_active_scale,
    get_active_voxel_size,
)
from deeptrack.image import Image, pad_image_to_fft


def _get_position(
    image: Image,
    mode: str = "corner",
    return_z: bool = False,
) -> np.ndarray:
    """Extracts the position of the upper-left corner of a scatterer.

    Parameters
    ----------
    image: numpy.ndarray
        Input image or volume containing the scatterer.
    mode: str, optional
        Mode for position extraction. Default is "corner".
    return_z: bool, optional
        Whether to include the z-coordinate in the output. Default is False.

    Returns
    -------
    numpy.ndarray
        Array containing the position of the scatterer.
    
    """

    num_outputs = 2 + return_z

    if mode == "corner" and image.size > 0:
        import scipy.ndimage

        image = image.to_numpy()

        shift = scipy.ndimage.center_of_mass(np.abs(image))

        if np.isnan(shift).any():
            shift = np.array(image.shape) / 2

    else:
        shift = np.zeros((num_outputs))

    position = np.array(image.get_property("position", default=None))

    if position is None:
        return position

    scale = np.array(get_active_scale())

    if len(position) == 3:
        position = position * scale + 0.5 * (scale - 1)
        if return_z:
            return position * scale - shift
        else:
            return position[0:2] - shift[0:2]

    elif len(position) == 2:
        if return_z:
            outp = (
                np.array([position[0], position[1], image.get_property("z", default=0)])
                * scale
                - shift
                + 0.5 * (scale - 1)
            )
            return outp
        else:
            return position * scale[:2] - shift[0:2] + 0.5 * (scale[:2] - 1)

    return position


# TODO ***??*** revise _create_volume - torch, typing, docstring, unit test
def _create_volume(
    list_of_scatterers: list,
    pad: tuple = (0, 0, 0, 0),
    output_region: tuple = (None, None, None, None),
    refractive_index_medium: float = 1.33,
    **kwargs: Any,
) -> tuple:
    """Converts a list of scatterers into a volumetric representation.

    Parameters
    ----------
    list_of_scatterers: list or single scatterer
        List of scatterers to include in the volume.
    pad: tuple of int, optional
        Padding for the volume in the format (left, right, top, bottom).
        Default is (0, 0, 0, 0).
    output_region: tuple of int, optional
        Region to output, defined as (x_min, y_min, x_max, y_max). Default is 
        None.
    refractive_index_medium: float, optional
        Refractive index of the medium surrounding the scatterers. Default is 
        1.33.
    **kwargs: Any
        Additional arguments for customization.

    Returns
    -------
    tuple
        - volume: numpy.ndarray
            The generated volume containing the scatterers.
        - limits: numpy.ndarray
            Spatial limits of the volume.

    """

    if not isinstance(list_of_scatterers, list):
        list_of_scatterers = [list_of_scatterers]

    volume = np.zeros((1, 1, 1), dtype=complex)
    limits = None
    OR = np.zeros((4,))
    OR[0] = -np.inf if output_region[0] is None else int(
        output_region[0] - pad[0]
    )
    OR[1] = np.inf if output_region[1] is None else int(
        output_region[1] - pad[1]
    )
    OR[2] = -np.inf if output_region[2] is None else int(
        output_region[2] + pad[2]
    )
    OR[3] = np.inf if output_region[3] is None else int(
        output_region[3] + pad[3]
    )

    scale = np.array(get_active_scale())

    # This accounts for upscale doing AveragePool instead of SumPool. This is
    # a bit of a hack, but it works for now.
    fudge_factor = scale[0] * scale[1] / scale[2]

    for scatterer in list_of_scatterers:

        position = _get_position(scatterer, mode="corner", return_z=True)

        if scatterer.get_property("intensity", None) is not None:
            intensity = scatterer.get_property("intensity")
            scatterer_value = intensity * fudge_factor
        elif scatterer.get_property("refractive_index", None) is not None:
            refractive_index = scatterer.get_property("refractive_index")
            scatterer_value = (
                refractive_index - refractive_index_medium
            )
        else:
            scatterer_value = scatterer.get_property("value")

        scatterer = scatterer * scatterer_value

        if limits is None:
            limits = np.zeros((3, 2), dtype=np.int32)
            limits[:, 0] = np.floor(position).astype(np.int32)
            limits[:, 1] = np.floor(position).astype(np.int32) + 1

        if (
            position[0] + scatterer.shape[0] < OR[0]
            or position[0] > OR[2]
            or position[1] + scatterer.shape[1] < OR[1]
            or position[1] > OR[3]
        ):
            continue

        padded_scatterer = Image(
            np.pad(
                scatterer,
                [(2, 2), (2, 2), (2, 2)],
                "constant",
                constant_values=0,
            )
        )
        padded_scatterer.merge_properties_from(scatterer)

        scatterer = padded_scatterer
        position = _get_position(scatterer, mode="corner", return_z=True)
        shape = np.array(scatterer.shape)

        if position is None:
            RuntimeWarning(
                "Optical device received an image without a position property."
                " It will be ignored."
            )
            continue

        splined_scatterer = np.zeros_like(scatterer)

        x_off = position[0] - np.floor(position[0])
        y_off = position[1] - np.floor(position[1])

        kernel = np.array(
            [
                [0, 0, 0],
                [0, (1 - x_off) * (1 - y_off), (1 - x_off) * y_off],
                [0, x_off * (1 - y_off), x_off * y_off],
            ]
        )

        for z in range(scatterer.shape[2]):
            if splined_scatterer.dtype == complex:
                splined_scatterer[:, :, z] = (
                    convolve(
                        np.real(scatterer[:, :, z]), kernel, mode="constant"
                    )
                    + convolve(
                        np.imag(scatterer[:, :, z]), kernel, mode="constant"
                    )
                    * 1j
                )
            else:
                splined_scatterer[:, :, z] = convolve(
                    scatterer[:, :, z], kernel, mode="constant"
                )

        scatterer = splined_scatterer
        position = np.floor(position)
        new_limits = np.zeros(limits.shape, dtype=np.int32)
        for i in range(3):
            new_limits[i, :] = (
                np.min([limits[i, 0], position[i]]),
                np.max([limits[i, 1], position[i] + shape[i]]),
            )

        if not (np.array(new_limits) == np.array(limits)).all():
            new_volume = np.zeros(
                np.diff(new_limits, axis=1)[:, 0].astype(np.int32),
                dtype=complex,
            )
            old_region = (limits - new_limits).astype(np.int32)
            limits = limits.astype(np.int32)
            new_volume[
                old_region[0, 0] : 
                old_region[0, 0] + limits[0, 1] - limits[0, 0],
                old_region[1, 0] : 
                old_region[1, 0] + limits[1, 1] - limits[1, 0],
                old_region[2, 0] : 
                old_region[2, 0] + limits[2, 1] - limits[2, 0],
            ] = volume
            volume = new_volume
            limits = new_limits

        within_volume_position = position - limits[:, 0]

        # NOTE: Maybe shouldn't be additive.
        volume[
            int(within_volume_position[0]) : 
            int(within_volume_position[0] + shape[0]),
            
            int(within_volume_position[1]) : 
            int(within_volume_position[1] + shape[1]),

            int(within_volume_position[2]) : 
            int(within_volume_position[2] + shape[2]),
        ] += scatterer
    return volume, limits

In [ ]:
V,L=_create_volume(particle)

In [ ]:
brightfield_microscope = dt.Darkfield(
    wavelength=500 * dt.units.nm, NA=1.0, resolution=1 * dt.units.um,
    magnification=1, refractive_index_medium=1.33, upsample=4,
    output_region=(0, 0, 64, 64),
)

In [ ]:
illuminated_sample = brightfield_microscope(particle)

In [ ]:
import matplotlib.pyplot as plt
def plot_image(title, image):
    """Plot a grayscale image with a title."""
    plt.imshow(image, cmap="gray")
    plt.title(title, fontsize=30)
    plt.axis("off")
    plt.show()

In [ ]:
plot_image('Illuminated Sample', illuminated_sample.resolve())

In [ ]:
import random
import numpy as np
import torch
# Set a fixed seed value
seed = 89

# Python, NumPy, and PyTorch (CPU)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Only set CUDA seeds if a GPU is available
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Seeds set to {seed} (with CUDA: {torch.cuda.is_available()})")


exp_crop_size = 30

# Same as when selecting a single object.
sim_crop_size = exp_crop_size

# Size of a pixel in nanometers in the output image.
pixel_size_nm = 100 # In nm.

# Size of the long ellipsoid semiaxis in nm.
major_axis_length = 1000 # In nm.

# Eccentricity of ellipsoid.
eccentricity = 0.185

ellipse = dt.Ellipsoid(
    position=0.5 * np.array([sim_crop_size, sim_crop_size]),
    z=0 * dt.units.nm, # Particle in focus.
    radius=(major_axis_length, eccentricity * major_axis_length) * dt.units.nm,  # Axes in nanometers
    intensity=0.35,  # Field magnitude squared
    rotation=0.225 * np.pi,
)

# Set the optical properties of the microscope.
optics = dt.Darkfield(
    NA=1.0,  # Numerical aperture
    wavelength=500 * dt.units.nm,
    refractive_index_medium=1.33,
    output_region=[0, 0, sim_crop_size, sim_crop_size],
    magnification=1,
    resolution=pixel_size_nm * dt.units.nm,  # Camera resolution or effective resolution.
    upscale=1,
)

# Apply transformations. Use `Upscale` to improve simulations rendering.
sim_crop = (
    optics(ellipse)
    # dt.Upscale(optics(ellipse), factor=2)  # Upscale the image to the original size.
    >> dt.Background(0)
    >> dt.Poisson(snr=40)
    >> dt.Multiply(70)
)

# Convert crop into NumPy array.
sim_crop = np.squeeze(sim_crop())

# Plot the simulated and experimental crops.
fig, axes = plt.subplots(1, 2)

# Simulated crop.
axes[0].imshow(sim_crop, cmap="gray")
axes[0].axis("off")
axes[0].set_title("Simulated Crop")

# # Experimental crop.
# axes[1].imshow(exp_crop, cmap="gray")
# axes[1].axis("off")
# axes[1].set_title("Experimental Crop")

# Adjust layout and show plot.
plt.tight_layout()
plt.show()